# Essai 2

Application du modèle Silero VAD pour préclassifier les extraits audios qui contiennent de la parole et ceux qui n'en contiennent pas.

In [ ]:
import json
from pathlib import Path
from google.colab import drive
import torch
import torchaudio
import torchaudio.transforms as T
from tqdm.notebook import tqdm

# 1. Montage du Drive
drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/Projet Multimedia/movie')
AUDIO_DIR = BASE_DIR / 'audio'
OUTPUT_VAD_FILE = BASE_DIR / 'vad_results.json'

# 2. Chargement du modèle Silero VAD
print('Chargement du modèle Silero VAD...')
model, utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False,
    onnx=False,
)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

TARGET_SR = 16000  # Silero VAD nécessite une fréquence de 16 kHz

def load_and_resample(audio_path):
    """Charge le fichier MP3, le passe en mono et le rééchantillonne à 16 kHz."""
    wav, sr = torchaudio.load(str(audio_path))

    # Conversion Stéréo -> Mono si nécessaire
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    # Rééchantillonnage vers 16 kHz
    if sr != TARGET_SR:
        resampler = T.Resample(sr, TARGET_SR)
        wav = resampler(wav)

    return wav.squeeze(0)  # Tensor 1D

# 3. Récupération de tous les audios MP3
audio_files = sorted(list(AUDIO_DIR.glob('*/*.mp3')))
print(f"Nombre total de fichiers audio à analyser : {len(audio_files)}")

vad_results = {}
speech_count = 0
non_speech_count = 0

# 4. Exécution du VAD
for audio_path in tqdm(audio_files, desc='Analyse VAD en cours'):
    clip_id = audio_path.stem      # ex: "00000_clip_001"
    film_id = audio_path.parent.name # ex: "00000"

    try:
        # Charger l'audio à 16kHz
        wav_tensor = load_and_resample(audio_path)

        # Obtenir les segments de parole
        speech_timestamps = get_speech_timestamps(
            wav_tensor,
            model,
            sampling_rate=TARGET_SR,
            threshold=0.5
        )

        # Calcul de la durée totale de parole détectée (en secondes)
        total_speech_samples = sum(ts['end'] - ts['start'] for ts in speech_timestamps)
        total_speech_duration = round(total_speech_samples / TARGET_SR, 2)

        has_speech = len(speech_timestamps) > 0 and total_speech_duration >= 0.5

        if has_speech:
            speech_count += 1
        else:
            non_speech_count += 1

        vad_results[clip_id] = {
            'film_id': film_id,
            'has_speech': has_speech,
            'speech_duration_sec': total_speech_duration,
            'speech_segments_count': len(speech_timestamps),
        }

    except Exception as e:
        print(f'\nErreur sur {clip_id}: {e}')
        vad_results[clip_id] = {
            'film_id': film_id,
            'has_speech': None,
            'error': str(e),
        }

# 5. Sauvegarde du récapitulatif
report = {
    'summary': {
        'total_clips': len(audio_files),
        'clips_with_speech': speech_count,
        'clips_without_speech': non_speech_count,
    },
    'results': vad_results,
}

with open(OUTPUT_VAD_FILE, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print('\n' + '=' * 40)
print(' BILAN VAD')
print('=' * 40)
print(f"Total des clips analysés : {len(audio_files)}")
print(f"Clips AVEC parole : {speech_count}")
print(f"Clips SANS parole (bruit/musique) : {non_speech_count}")
print(f'\nRésultats enregistrés dans : {OUTPUT_VAD_FILE}')

Mounted at /content/drive
Chargement du modèle Silero VAD...
The repository snakers4_silero-vad does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
Nombre total de fichiers audio à analyser : 3088


Analyse VAD en cours:   0%|          | 0/3088 [00:00<?, ?it/s]


 BILAN VAD
Total des clips analysés : 3088
Clips AVEC parole : 1971
Clips SANS parole (bruit/musique) : 1117

Résultats enregistrés dans : /content/drive/MyDrive/Projet Multimedia/movie/vad_results.json


Etape 2: Transcription de la parole

In [ ]:
!pip install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 27.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=6fe22f2f43eb086452a18823beda7c62c88949e7468088d7a92f0a124bd786e8
  Stored in directory: /root/.cache/pip/wheels/ca/58/d5/fb4539ad74c3ca81eb40f7eda020ac77d080b33ad57449d485
Successfully built openai-whisper


In [ ]:
import json
from pathlib import Path
from google.colab import drive
import torch
import torchaudio
import torchaudio.transforms as T
from tqdm.notebook import tqdm
import whisper

# 1. Montage du Google Drive
drive.mount("/content/drive")

# 2. Chemins
BASE_DIR = Path("/content/drive/MyDrive/Projet Multimedia/movie")
AUDIO_DIR = BASE_DIR / "audio"
TRANSCRIPT_DIR = BASE_DIR / "transcriptionsVAD"
VAD_FILE = BASE_DIR / "vad_results.json"

if not VAD_FILE.exists():
    raise FileNotFoundError(
        f"Le fichier {VAD_FILE} est introuvable. Vérifiez que l'étape 1 (VAD) a bien été exécutée."
    )

with open(VAD_FILE, "r", encoding="utf-8") as f:
    vad_data = json.load(f)

vad_results = vad_data.get("results", {})
print(f"Nombre total de clips à traiter : {len(vad_results)}")

# 3. Chargement du modèle OpenAI Whisper (PyTorch, GPU si disponible)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Chargement du modèle OpenAI Whisper (small) sur le périphérique : {device}")
model = whisper.load_model("small", device=device)

TARGET_SR = 16000  # Fréquence d'échantillonnage attendue par Whisper

def load_audio_as_tensor(audio_path: Path):
    """Charge le fichier MP3 avec torchaudio, le convertit en mono et le rééchantillonne à 16 kHz."""
    wav, sr = torchaudio.load(str(audio_path))

    # Conversion en mono si nécessaire
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    # Rééchantillonnage à 16 kHz
    if sr != TARGET_SR:
        resampler = T.Resample(sr, TARGET_SR)
        wav = resampler(wav)

    # Whisper accepte directement un tenseur 1D en float32
    return wav.squeeze(0)

# 4. Traitement des clips
transcribed_count = 0      # Clips avec parole transcrits pendant cette exécution
no_speech_count = 0        # Clips sans parole (VAD) enregistrés avec une transcription vide
already_done_count = 0     # Clips déjà traités lors d'une exécution précédente (reprise)
missing_audio_count = 0    # Fichiers audio introuvables
error_count = 0            # Clips ayant provoqué une erreur de transcription

for clip_id, vad_info in tqdm(vad_results.items(), desc="Transcription des audios (OpenAI Whisper)"):
    film_id = vad_info.get("film_id")
    has_speech = vad_info.get("has_speech", False)

    target_film_dir = TRANSCRIPT_DIR / film_id
    target_film_dir.mkdir(parents=True, exist_ok=True)
    output_json_path = target_film_dir / f"{clip_id}.json"

    # Reprise automatique : on ignore les clips déjà traités
    if output_json_path.exists():
        already_done_count += 1
        continue

    audio_path = AUDIO_DIR / film_id / f"{clip_id}.mp3"
    if not audio_path.exists():
        missing_audio_count += 1
        continue

    # CAS 1 : aucune parole détectée par Silero VAD -> transcription vide
    if not has_speech:
        no_speech_count += 1
        data = {
            "video_id": clip_id,
            "transcript": "",
            "has_speech": False,
            "model": "openai-whisper-small",
        }
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        continue

    # CAS 2 : parole détectée -> transcription avec OpenAI Whisper
    try:
        audio_tensor = load_audio_as_tensor(audio_path)

        # Lancement de la transcription (langue détectée automatiquement)
        result = model.transcribe(
            audio_tensor,
            language=None,
            fp16=(device == "cuda"),
            no_speech_threshold=0.6,
            logprob_threshold=-1.0,
            compression_ratio_threshold=2.4,
            condition_on_previous_text=False,   # évite que une hallucination se propage au segment suivant
        )
        full_text = result.get("text", "").strip()

        # Langue détectée par Whisper
        detected_language = result.get("language", "unknown")

        data = {
            "video_id": clip_id,
            "transcript": full_text,
            "has_speech": True,
            "language": detected_language,
            "model": "openai-whisper-small",
        }

        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        transcribed_count += 1

    except Exception as e:
        error_count += 1
        print(f"\nErreur lors du traitement de {clip_id} : {e}")

# 5. Bilan
print("\n" + "=" * 50)
print(" BILAN DE LA TRANSCRIPTION (OpenAI Whisper small)")
print("=" * 50)
print(f"Clips au total                                 : {len(vad_results)}")
print(f"Clips avec parole transcrits (cette exécution) : {transcribed_count}")
print(f"Clips sans parole (transcription vide)         : {no_speech_count}")
print(f"Clips déjà traités auparavant (ignorés)        : {already_done_count}")
print(f"Clips dont l'audio est introuvable             : {missing_audio_count}")
print(f"Clips en erreur                                : {error_count}")

Mounted at /content/drive
Nombre total de clips à traiter : 3088
Chargement du modèle OpenAI Whisper (small) sur le périphérique : cuda


100%|████████████████████████████████████████| 461M/461M [00:02<00:00, 232MiB/s]


Transcription des audios (OpenAI Whisper):   0%|          | 0/3088 [00:00<?, ?it/s]


 BILAN DE LA TRANSCRIPTION (OpenAI Whisper small)
Clips au total                                 : 3088
Clips avec parole transcrits (cette exécution) : 0
Clips sans parole (transcription vide)         : 0
Clips déjà traités auparavant (ignorés)        : 3088
Clips dont l'audio est introuvable             : 0
Clips en erreur                                : 0


In [ ]:
import json
import csv
from collections import Counter
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/Projet Multimedia")
ANNOTATION_DIR = BASE_DIR / "movie/annotation"
TRANSCRIPT_DIR = BASE_DIR / "movie/transcriptionsVAD"
OUTPUT_CSV = BASE_DIR / "dataset.csv"

# Encodage numérique des émotions (ordre demandé, indices à partir de 0)
EMOTIONS = ["amusement", "awe", "contentment", "excitement",
            "anger", "disgust", "fear", "sadness"]
EMOTION_TO_ID = {emo: i for i, emo in enumerate(EMOTIONS)}
EMOTION_TO_ID["amusment"] = EMOTION_TO_ID["amusement"]  # variante orthographique

print("Encodage :", {e: i for i, e in enumerate(EMOTIONS)})

# --- 1. Lecture des partitions (IDs des films) ---
def load_ids(path):
    with open(path, "r", encoding="utf-8") as f:
        return {line.strip().zfill(5) for line in f if line.strip()}

film_to_split = {}
for split_name in ["train", "dev", "test"]:
    ids_file = BASE_DIR / f"split/IDs_{split_name}.txt"
    if not ids_file.exists():
        found = list(BASE_DIR.rglob(f"IDs_{split_name}.txt"))
        if not found:
            raise FileNotFoundError(f"IDs_{split_name}.txt introuvable dans {BASE_DIR}")
        ids_file = found[0]
    ids = load_ids(ids_file)
    for fid in ids:
        film_to_split[fid] = split_name
    print(f"{split_name}: {len(ids)} films")

# --- 2. Construction des lignes du CSV ---
rows = []
missing_transcripts = 0
missing_labels = 0
no_split = 0
unknown_emotions = Counter()

for ann_path in sorted(ANNOTATION_DIR.glob("*/*.json")):
    video_id = ann_path.stem            # ex: 00103_clip_001
    film_id = video_id.split("_")[0]    # ex: 00103

    with open(ann_path, "r", encoding="utf-8") as f:
        ann = json.load(f)

    caption = ann.get("caption", "")

    # Emotion -> entier
    emotion_raw = str(ann.get("emotion", "")).strip().lower()
    if emotion_raw in EMOTION_TO_ID:
        label = EMOTION_TO_ID[emotion_raw]
    else:
        label = ""
        missing_labels += 1
        if emotion_raw:
            unknown_emotions[emotion_raw] += 1

    split = film_to_split.get(film_id, "")
    if split == "":
        no_split += 1

    # Transcription (produite à l'étape 2)
    tr_path = TRANSCRIPT_DIR / film_id / f"{video_id}.json"
    if tr_path.exists():
        with open(tr_path, "r", encoding="utf-8") as f:
            tr = json.load(f)
        transcript = tr.get("transcript", "")
        has_speech = bool(tr.get("has_speech", False))
    else:
        missing_transcripts += 1
        transcript = ""
        has_speech = False

    rows.append({
        "video_id": video_id,
        "film_id": film_id,
        "label": label,
        "split": split,
        "caption": caption,
        "transcript": transcript,
        "has_speech": has_speech,
    })

# --- 3. Écriture du CSV ---
fieldnames = ["video_id", "film_id", "label", "split", "caption", "transcript", "has_speech"]
with open(OUTPUT_CSV, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\n" + "=" * 40)
print(f"CSV enregistré : {OUTPUT_CSV}")
print(f"Lignes écrites           : {len(rows)}")
print(f"Sans transcription       : {missing_transcripts}")
print(f"Sans label valide        : {missing_labels}")
print(f"Sans split (film absent) : {no_split}")
if unknown_emotions:
    print(f"Émotions non reconnues   : {dict(unknown_emotions)}")

# Répartition des labels par split, pour vérification
import pandas as pd
df = pd.DataFrame(rows)
print("\nRépartition label x split :")
print(pd.crosstab(df["label"], df["split"]))

Encodage : {'amusement': 0, 'awe': 1, 'contentment': 2, 'excitement': 3, 'anger': 4, 'disgust': 5, 'fear': 6, 'sadness': 7}
train: 250 films
dev: 50 films
test: 50 films

CSV enregistré : /content/drive/MyDrive/Projet Multimedia/dataset.csv
Lignes écrites           : 3089
Sans transcription       : 1
Sans label valide        : 0
Sans split (film absent) : 0

Répartition label x split :
split  dev  test  train
label                  
0       35    35    174
1       40    40    206
2       54    54    265
3       51    51    252
4       98    96    484
5       43    43    211
6       74    74    371
7       47    46    245
